# LoRA: Low-Rank Adaptation of Large Language Models
### Annotated Implementation

**Hu et al., 2021 · [arXiv:2106.09685](https://arxiv.org/abs/2106.09685)**  
Edward Hu, Yelong Shen, Phillip Wallis, Zeyuan Allen-Zhu, Yuanzhi Li, Shean Wang, Lu Wang, Weizhu Chen · Microsoft Corporation

---

This notebook is an annotated walkthrough of the LoRA paper, in the style of the
[Harvard Annotated Transformer](https://nlp.seas.harvard.edu/2018/04/03/attention.html).


### Background

Many applications in modern natural language processing need one massive general trained model to adapt a diverse class of problems downstream. Such adaptations are done by fine-tuning. Which is the process that updates all the pre-trained model parameters specifically for the new task. The biggest problem is fine-tuning trains the same amount of parameters as the original mode, which leads to a serious amount of cost with each fine-tune for different applications. As models are getting larger and larger with more parameters and tokens, this problem increases to a critical point hard to ignore. 

There were multiple different mitigations for a solution but they often introduced inference latency. Also, these methods fail to cover the fine-tuning baseline leading to a trade-off between quality and productiveness.

### Paper's Solution

They hypothize most models are over-parametrized, so they reside on a low intrinsic dimension. Low intrinsic meaning that even though LLM's have huge weight matrices, the actual meaningful changes during training or fine-tuning lie in a much lower-dimensional space. And the paper could capture this meaning lying in lower dimension with optimizing rank decomposition matrices(A and B) of some selected layers, while keeping the original training weights frozen/untouched. There are several advantages to this strategy: 

• An already pre-trained model can be shared with different applications for 
different tasks, only changing its A and B matrices, thus reducing the 
storage requirement and task-switching over-head significantly.

• This makes training  efficient and lowers the hardware requirements by up to 3
times when using adaptive optimizers since they only optimize the injected,
much smaller low-rank matrices.

• The simple linear design allows to merge the trainable matrices with the frozen weights, 
thus leading to no inference latency unlike other methods.

• LoRA could be combined with many of the other methods, such
as prefix-tuning. 

In [2]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset

DEVICE = "cpu"
print("PyTorch:", torch.__version__, "| Device:", DEVICE)

PyTorch: 2.8.0+cpu | Device: cpu


## Problem Statement

### Full fine-tuning (Equation 1)

$$\max_{\Phi} \sum_{(x,y)\in\mathcal{Z}} \sum_{t=1}^{|y|} \log P_{\Phi}(y_t \mid x,\, y_{<t})$$

| Symbol | Meaning |
|--------|---------|
| $\Phi$ | **All** model parameters (175B for GPT-3). The full thing is updated. |
| $(x,y) \in \mathcal{Z}$ | A (context, target) pair. For NL→SQL: $x$ = question, $y$ = SQL. For summarisation: $x$ = article, $y$ = summary. |
| $y_t$ | The $t$-th target token. |
| $y_{<t}$ | All target tokens before position $t$ — autoregressive conditioning. |
| $\log P_\Phi$ | Log-probability of the correct token. Maximising = minimising cross-entropy. |

### LoRA objective (Equation 2)

$$\max_{\Theta} \sum_{(x,y)\in\mathcal{Z}} \sum_{t=1}^{|y|} \log p_{\Phi_0 + \Delta\Phi(\Theta)}(y_t \mid x,\, y_{<t})$$

| Symbol | Meaning |
|--------|---------|
| $\Theta$ | The **small** LoRA parameters. For GPT-3, $|\Theta| \approx 0.01\%$ of $|\Phi_0|$. |
| $\Phi_0$ | Frozen pre-trained weights — shared across all tasks, never updated. |
| $\Delta\Phi(\Theta)$ | The update, now expressed via a low-rank factorisation. Same shape as $\Phi_0$, determined by far fewer free numbers. |

> **Key:** The *loss function* is identical to full fine-tuning. Only the *parametrisation*
> of the update changes — a representation constraint, not a change to the objective.

In [ ]:
class LoraLayer(nn.Module):
    def __init__(self, in_features: int, out_features: int,
                 r: int = 4, alpha: float = 1.0, dropout: float = 0.0):
        super().__init__()
        self.r     = r
        self.scale = alpha / r

        self.weight = nn.Parameter(torch.empty(out_features, in_features), requires_grad=False)
        self.bias   = nn.Parameter(torch.zeros(out_features), requires_grad=False)

        self.lora_A = nn.Parameter(torch.empty(self.r, in_features))
        self.lora_B = nn.Parameter(torch.empty(out_features, self.r))

        self.dropout = nn.Dropout(p=dropout) if dropout > 0 else nn.Identity()
        self._reset_parameters()

    def _reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        nn.init.zeros_(self.bias)
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base = F.linear(x, self.weight, self.bias)
        lora = F.linear(F.linear(self.dropout(x), self.lora_A), self.lora_B) * self.scale
        return base + lora

    def mergeweights(self) -> None:
        with torch.no_grad():
            self.weight.data += (self.lora_B @ self.lora_A) * self.scale

    def extra_repr(self):
        return (f"in={self.weight.shape[1]}, out={self.weight.shape[0]}, "
                f"r={self.r}, scale={self.scale:.3f}")

In [ ]:
import urllib.request
from datasets import Dataset
from transformers import GPT2Tokenizer

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")

with open("shakespeare.txt", "r", encoding="utf-8") as f:
    lines = [l.strip() for l in f if l.strip()]

entries = lines[:200]

d = Dataset.from_dict({"text": entries})

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  

tok = tokenizer(entries, padding=True, truncation=True, return_tensors="pt")
print(tok["input_ids"].shape)  


torch.Size([200, 16])
['First Citizen:', 'Before we proceed any further, hear me speak.', 'All:', 'Speak, speak.', 'First Citizen:', 'You are all resolved rather to die than to famish?', 'All:', 'Resolved. resolved.', 'First Citizen:', 'First, you know Caius Marcius is chief enemy to the people.', 'All:', "We know't, we know't.", 'First Citizen:', "Let us kill him, and we'll have corn at our own price.", "Is't a verdict?", 'All:', "No more talking on't; let it be done: away, away!", 'Second Citizen:', 'One word, good citizens.', 'First Citizen:', 'We are accounted poor citizens, the patricians good.', 'What authority surfeits on would relieve us: if they', 'would yield us but the superfluity, while it were', 'wholesome, we might guess they relieved us humanely;', 'but they think we are too dear: the leanness that', 'afflicts us, the object of our misery, is as an', 'inventory to particularise their abundance; our', 'sufferance is a gain to them Let us revenge this with', 'our pikes, er

In [ ]:
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.requires_grad_(False)  # freeze all pretrained weights

for head in model.transformer.h[-3:]:
    orig_c_attn = head.attn.c_attn
    orig_c_proj = head.attn.c_proj

    head.attn.c_attn = LoraLayer(orig_c_attn.weight.shape[0], orig_c_attn.weight.shape[1], r=4, alpha=1.0)
    head.attn.c_proj = LoraLayer(orig_c_proj.weight.shape[0], orig_c_proj.weight.shape[1], r=4, alpha=1.0)

    head.attn.c_attn.weight.data.copy_(orig_c_attn.weight.t())
    head.attn.c_proj.weight.data.copy_(orig_c_proj.weight.t())
    head.attn.c_attn.bias.data.copy_(orig_c_attn.bias)
    head.attn.c_proj.bias.data.copy_(orig_c_proj.bias)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")